# PersonaRAG — Corpus-Conditioned Benchmark Evaluation

**Datasets:** HotpotQA (distractor) · Natural Questions Open · TriviaQA (Wikipedia)  
**Baselines:** `dense_only` · `bm25_only` · `hybrid` · `hybrid_rerank`  
**Metrics:** Exact Match (EM) · Token F1 · Recall@K · Support Rate · Latency  
**Generator:** Llama-3.1-8B-Instruct loaded locally in 4-bit (no API key required)

---

### Evaluation Methodology

This notebook implements **corpus-conditioned evaluation** — the standard methodology used in open-domain QA research (DPR, RAG, FiD).

Rather than retrieval from a fixed personal index, each question is evaluated against **its own passage corpus drawn from the dataset itself**:

| Dataset | Corpus per question | Gold signal |
|---|---|---|
| **HotpotQA** (distractor) | 10 Wikipedia paragraphs (2 gold + 8 hard distractors) | `supporting_facts` titles |
| **TriviaQA** (rc.wikipedia) | Evidence Wikipedia passages per question | Answer string present in passage |
| **NQ Open** | No per-question passages in HF release | Generation-only (EM/F1); Recall@K = N/A |

**Retrieval evaluation:** For HotpotQA and TriviaQA, each retriever is given only the question's own passage pool. Recall@K measures whether the gold passage reaches the top-K retrieved set — isolating retrieval quality from corpus size effects.

**Generation evaluation:** Retrieved passages are passed to **Llama-3.1-8B-Instruct loaded locally** via `transformers` 4-bit quantisation. No external API calls — runs entirely on the Colab T4 GPU.

> Run this notebook top-to-bottom in Google Colab with a T4 GPU runtime.


## Step 1 — Runtime Check

In [ ]:
import sys, platform, subprocess
print("Python:", sys.version)
print("Platform:", platform.platform())

# Check for GPU
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                            capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("GPU:", result.stdout.strip())
    else:
        print("GPU: Not available — CPU mode. Eval will be slower but will still run.")
except Exception:
    print("GPU: Not available — CPU mode.")


## Step 2 — Clone Repo and Install Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/ExperimenterX/PersonaRAG.git"
REPO_DIR = "/content/PersonaRAG"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest.")
    !cd {REPO_DIR} && git pull

os.chdir(f"{REPO_DIR}/server")
print("Working dir:", os.getcwd())


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# NOTE: Do NOT install numpy or torch here.
# Colab ships pre-built, ABI-matched versions. Overriding them causes binary
# incompatibility errors in faiss, numpy.random, etc.

!pip install -q \
    "sentence-transformers>=3.0.0" \
    "faiss-cpu" \
    "rank-bm25"

# Local model loading (transformers pipeline — no external API needed)
!pip install -q "transformers>=4.43.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

# Dataset loader + plotting
!pip install -q "huggingface_hub>=0.23.0" "datasets>=2.20.0"
!pip install -q matplotlib pandas "tqdm>=4.65"

print("Dependencies installed.")
print()
print(">>> IMPORTANT: Go to Runtime → Restart session, then re-run from Step 3. <<<")


## Step 3 — Set API Keys

In [ ]:
import os
import torch
from huggingface_hub import login

# ── HF token ──────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    if _token:
        os.environ["HF_TOKEN"] = _token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("WARNING: HF_TOKEN secret is empty — ungated models (Qwen) will still work.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN loaded from environment.")
    else:
        print("WARNING: No HF_TOKEN found — ungated models will still work without it.")

# Optional login — only needed for gated models (Llama). Skip if token is missing/invalid.
_hf_token = os.environ.get("HF_TOKEN", "")
if _hf_token:
    try:
        login(token=_hf_token, add_to_git_credential=False)
        print("huggingface_hub login OK.")
    except Exception as _login_err:
        print(f"WARNING: huggingface_hub login failed ({_login_err}).")
        print("  Ungated models (Qwen2.5, Phi-3.5) will still download fine.")
        print("  To use gated Llama models, fix the token in Colab Secrets.")

# ── Retrieval hyper-parameters ────────────────────────────────────────────────
K_DENSE  = 10
K_BM25   = 10
K_RERANK = 5
ALPHA    = 0.65

# ── Generator config — local only, no external API ───────────────────────────
# All models below are loaded in-process via transformers pipeline (4-bit on GPU).
#
#   local_qwen25    → Qwen/Qwen2.5-7B-Instruct          NO license gate  (~5 GB VRAM 4-bit, T4 OK)
#   local_llama3    → meta-llama/Meta-Llama-3.1-8B-Instruct  gated — must accept license first
#   local_llama3_3b → meta-llama/Llama-3.2-3B-Instruct  gated — must accept license first
#   local_phi35     → microsoft/Phi-3.5-mini-instruct    NO license gate  (~2 GB, CPU-feasible)

GENERATOR_BACKEND = "local_qwen25"   # ← default: no gate, no token needed

GENERATOR_CONFIGS = {
    "local_qwen25": {
        "model":    "Qwen/Qwen2.5-7B-Instruct",
        "provider": "local",
        "label":    "Qwen2.5-7B-Instruct (local 4-bit)",
        "gated":    False,
    },
    "local_llama3": {
        "model":    "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "provider": "local",
        "label":    "Llama-3.1-8B-Instruct (local 4-bit)",
        "gated":    True,
    },
    "local_llama3_3b": {
        "model":    "meta-llama/Llama-3.2-3B-Instruct",
        "provider": "local",
        "label":    "Llama-3.2-3B-Instruct (local 4-bit)",
        "gated":    True,
    },
    "local_phi35": {
        "model":    "microsoft/Phi-3.5-mini-instruct",
        "provider": "local",
        "label":    "Phi-3.5-mini-Instruct (local 4-bit)",
        "gated":    False,
    },
}

_hf_cfg = GENERATOR_CONFIGS[GENERATOR_BACKEND]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Device    : {DEVICE}  (dtype={DTYPE})")
print(f"Generator : {_hf_cfg['label']}")
print(f"K_dense={K_DENSE}  K_BM25={K_BM25}  K_rerank={K_RERANK}  alpha={ALPHA}")

if _hf_cfg["gated"] and not _hf_token:
    raise EnvironmentError(
        f"{_hf_cfg['model']} is a gated repo. "
        "A valid HF_TOKEN is required. Switch to 'local_qwen25' or fix the token."
    )


## Step 4 — Scoring Utilities
Normalised EM, token F1, and Recall@K — identical to metrics used in DPR / RAG literature.


In [ ]:
import string, collections
from typing import List, Dict, Any, Optional


# ── Text normalisation (matches DPR / SQuAD official scripts) ───────────────

def normalize_text(text: str) -> str:
    text = (text or "").lower().strip()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    return " ".join(text.split())


# ── Exact Match ──────────────────────────────────────────────────────────────

def exact_match(prediction: str, gold_answers: List[str]) -> float:
    pred = normalize_text(prediction)
    return 1.0 if any(pred == normalize_text(g) for g in gold_answers if g) else 0.0


# ── Token F1 ─────────────────────────────────────────────────────────────────

def token_f1(prediction: str, gold_answers: List[str]) -> float:
    pred_tokens = normalize_text(prediction).split()
    if not pred_tokens or not gold_answers:
        return 0.0
    best = 0.0
    pred_counts = collections.Counter(pred_tokens)
    for gold in gold_answers:
        gold_tokens = normalize_text(gold).split()
        if not gold_tokens:
            continue
        gold_counts = collections.Counter(gold_tokens)
        common = sum((pred_counts & gold_counts).values())
        if common == 0:
            continue
        p = common / len(pred_tokens)
        r = common / len(gold_tokens)
        best = max(best, 2 * p * r / (p + r))
    return best


# ── Recall@K ─────────────────────────────────────────────────────────────────

def recall_at_k(
    retrieved_ids: List[str],
    gold_ids: List[str],       # passage IDs that are considered gold
    k: int,
) -> float:
    """1.0 if at least one gold passage ID appears in the top-k, else 0.0."""
    if not gold_ids:
        return float("nan")   # unanswerable / no gold known
    topk = set(retrieved_ids[:k])
    return 1.0 if any(gid in topk for gid in gold_ids) else 0.0


# ── Support Rate (lexical answer-in-context) ──────────────────────────────────

def support_rate(answers: List[str], context_texts: List[str]) -> float:
    if not answers or not context_texts:
        return 0.0
    joined = " ".join(context_texts).lower()
    return 1.0 if any(a.strip().lower() in joined for a in answers if a) else 0.0


print("Scoring utilities loaded.")


## Step 5 — Load Embedding Model & Retrieval Helpers
All baselines share the same encoder (`intfloat/e5-base-v2`, matching PersonaRAG's config). Retrieval helpers build **ephemeral per-question indices** — no pre-built personal corpus is involved.


In [ ]:
import os, sys, time
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

EMBED_MODEL_NAME = "intfloat/e5-base-v2"   # same as PersonaRAG config
K_DENSE   = 10       # top-K for dense retrieval
K_BM25    = 10       # top-K for BM25
K_RERANK  = 5        # final top-K after rerank / fusion
ALPHA     = 0.65     # hybrid fusion weight (dense fraction), same as config

print(f"Loading embedding model: {EMBED_MODEL_NAME} ...")
ENCODER = SentenceTransformer(EMBED_MODEL_NAME)
print("Encoder ready.")


# ── Per-question dense index ──────────────────────────────────────────────────

def build_faiss_index(passages: List[Dict[str, Any]]) -> faiss.IndexFlatIP:
    """Encode passages and build a flat inner-product (cosine) FAISS index."""
    texts = [f"passage: {p['text']}" for p in passages]
    embeddings = ENCODER.encode(texts, normalize_embeddings=True,
                                show_progress_bar=False)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))
    return index


def dense_retrieve(query: str, passages: List[Dict[str, Any]],
                   index: faiss.IndexFlatIP, k: int) -> List[Dict[str, Any]]:
    q_emb = ENCODER.encode([f"query: {query}"], normalize_embeddings=True)
    scores, ids = index.search(q_emb.astype(np.float32), min(k, len(passages)))
    return [{"passage": passages[i], "score": float(scores[0][rank])}
            for rank, i in enumerate(ids[0]) if i >= 0]


# ── BM25 retrieval ────────────────────────────────────────────────────────────

def bm25_retrieve(query: str, passages: List[Dict[str, Any]],
                  k: int) -> List[Dict[str, Any]]:
    corpus = [p["text"].lower().split() for p in passages]
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(query.lower().split())
    top_ids = np.argsort(scores)[::-1][:k]
    return [{"passage": passages[i], "score": float(scores[i])} for i in top_ids]


# ── Hybrid fusion ─────────────────────────────────────────────────────────────

def _normalize_scores(hits: List[Dict]) -> List[Dict]:
    if not hits:
        return hits
    vals = [h["score"] for h in hits]
    mn, mx = min(vals), max(vals)
    rng = mx - mn if mx != mn else 1.0
    return [{**h, "score": (h["score"] - mn) / rng} for h in hits]


def hybrid_retrieve(query: str, passages: List[Dict[str, Any]],
                    index: faiss.IndexFlatIP, k: int,
                    alpha: float = ALPHA) -> List[Dict[str, Any]]:
    dense_hits = _normalize_scores(dense_retrieve(query, passages, index, len(passages)))
    bm25_hits  = _normalize_scores(bm25_retrieve(query, passages, len(passages)))

    score_map: Dict[str, float] = {}
    id_to_passage: Dict[str, Any] = {}

    for h in dense_hits:
        pid = h["passage"]["id"]
        score_map[pid] = score_map.get(pid, 0.0) + alpha * h["score"]
        id_to_passage[pid] = h["passage"]
    for h in bm25_hits:
        pid = h["passage"]["id"]
        score_map[pid] = score_map.get(pid, 0.0) + (1.0 - alpha) * h["score"]
        id_to_passage[pid] = h["passage"]

    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)[:k]
    return [{"passage": id_to_passage[pid], "score": s} for pid, s in ranked]


# ── Reranker (cross-encoder via sentence-transformers) ────────────────────────
# Uses BAAI/bge-reranker-base — a real cross-encoder, not a stub.

from sentence_transformers import CrossEncoder

print("Loading cross-encoder reranker (BAAI/bge-reranker-base) ...")
RERANKER = CrossEncoder("BAAI/bge-reranker-base")
print("Reranker ready.")


def rerank(query: str, hits: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:
    if not hits:
        return hits
    pairs = [(query, h["passage"]["text"]) for h in hits]
    scores = RERANKER.predict(pairs)
    reranked = sorted(zip(hits, scores), key=lambda x: x[1], reverse=True)
    return [{"passage": h["passage"], "score": float(s)} for h, s in reranked[:k]]


print("\nAll retrieval helpers ready.")
print(f"  Encoder : {EMBED_MODEL_NAME}")
print(f"  Reranker: BAAI/bge-reranker-base")
print(f"  K_dense={K_DENSE}, K_bm25={K_BM25}, K_rerank={K_RERANK}, alpha={ALPHA}")


## Step 6 — Configure & Load Datasets

Each dataset loader returns a list of `EvalExample` objects.  

Every `EvalExample` carries its own `passages` list (the question's corpus) plus the `gold_passage_ids` used to compute Recall@K.

| Dataset | HF identifier | Passages field | Gold signal |
|---|---|---|---|
| HotpotQA | `hotpot_qa / distractor` | `context` (10 paragraphs) | `supporting_facts` titles |
| TriviaQA | `trivia_qa / rc.wikipedia` | `entity_pages.wiki_context` | answer string in passage |
| NQ Open | `nq_open` | — (no passages) | answer string only |


In [ ]:
from dataclasses import dataclass, field
from datasets import load_dataset

LIMIT = 200   # samples per dataset (use 500+ for publication; 200 is reliable on Colab free tier)
SPLIT = "validation"

MODES = ["dense_only", "bm25_only", "hybrid", "hybrid_rerank"]

OUTPUT_DIR = Path("/content/PersonaRAG/server/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Limit : {LIMIT} samples per dataset")
print(f"Split : {SPLIT}")
print(f"Modes : {MODES}")
print(f"Output: {OUTPUT_DIR}")


@dataclass
class EvalExample:
    qid:             str
    question:        str
    answers:         List[str]
    passages:        List[Dict[str, Any]]   # [{id, text}] — the per-question corpus
    gold_passage_ids: List[str]             # passage IDs considered gold for Recall@K
    has_corpus:      bool = True            # False for NQ (no per-question passages)


# ── HotpotQA (distractor setting) ────────────────────────────────────────────

def load_hotpotqa(limit: int, split: str) -> List[EvalExample]:
    print("Loading HotpotQA (distractor) ...")
    ds = load_dataset("hotpot_qa", "distractor", split=split,
                      trust_remote_code=True).select(range(min(limit, 7405)))
    examples = []
    for row in ds:
        # Build passage pool: concatenate sentences per paragraph
        passages = []
        for title, sents in zip(row["context"]["title"], row["context"]["sentences"]):
            passages.append({"id": title, "text": " ".join(sents)})

        # Gold: titles that appear in supporting_facts
        gold_titles = list(dict.fromkeys(row["supporting_facts"]["title"]))  # dedup, preserve order

        examples.append(EvalExample(
            qid=row["id"],
            question=row["question"],
            answers=[row["answer"]],
            passages=passages,
            gold_passage_ids=gold_titles,
            has_corpus=True,
        ))
    print(f"  Loaded {len(examples)} HotpotQA examples.")
    return examples


# ── TriviaQA (rc.wikipedia) ───────────────────────────────────────────────────

def load_triviaqa(limit: int, split: str) -> List[EvalExample]:
    print("Loading TriviaQA (rc.wikipedia) ...")
    ds = load_dataset("trivia_qa", "rc.wikipedia", split=split,
                      trust_remote_code=True).select(range(min(limit, 11313)))
    examples = []
    for row in ds:
        # Each question has a list of Wikipedia evidence passages
        wiki_pages  = row.get("entity_pages", {})
        wiki_texts  = wiki_pages.get("wiki_context", []) or []
        wiki_titles = wiki_pages.get("title", []) or []

        passages = [
            {"id": f"trivia_{row['question_id']}_{i}", "text": text}
            for i, text in enumerate(wiki_texts) if text
        ]

        # Gold: passages that contain at least one answer alias
        all_aliases = []
        ans = row.get("answer", {})
        if isinstance(ans, dict):
            all_aliases += ans.get("aliases", []) or []
            all_aliases += ans.get("normalized_aliases", []) or []
            val = ans.get("value") or ans.get("normalized_value")
            if val:
                all_aliases.append(val)

        gold_ids = [
            p["id"] for p in passages
            if any(alias.lower() in p["text"].lower() for alias in all_aliases if alias)
        ]

        answers = all_aliases[:1] or []   # primary answer for scoring

        examples.append(EvalExample(
            qid=row["question_id"],
            question=row["question"],
            answers=all_aliases,
            passages=passages,
            gold_passage_ids=gold_ids,
            has_corpus=bool(passages),
        ))
    examples = [e for e in examples if e.passages]   # skip questions with no evidence
    print(f"  Loaded {len(examples)} TriviaQA examples (with passages).")
    return examples


# ── NQ Open ───────────────────────────────────────────────────────────────────

def load_nq(limit: int, split: str) -> List[EvalExample]:
    print("Loading NQ Open ...")
    # nq_open validation split has ~3610 examples
    ds = load_dataset("nq_open", split=split,
                      trust_remote_code=True).select(range(min(limit, 3610)))
    examples = []
    for i, row in enumerate(ds):
        ans = row.get("answer", [])
        if isinstance(ans, str):
            ans = [ans]
        examples.append(EvalExample(
            qid=f"nq_{i}",
            question=row["question"],
            answers=ans,
            passages=[],          # no per-question corpus available
            gold_passage_ids=[],
            has_corpus=False,
        ))
    print(f"  Loaded {len(examples)} NQ Open examples (generation-only — no corpus).")
    return examples


# ── Load all datasets ─────────────────────────────────────────────────────────

DATASET_LOADERS = {
    "hotpotqa": load_hotpotqa,
    "triviaqa":  load_triviaqa,
    "nq":        load_nq,
}

ALL_EXAMPLES: Dict[str, List[EvalExample]] = {}
for name, loader in DATASET_LOADERS.items():
    ALL_EXAMPLES[name] = loader(LIMIT, SPLIT)

print("\nDataset summary:")
for name, exs in ALL_EXAMPLES.items():
    has = sum(e.has_corpus for e in exs)
    print(f"  {name:12s}: {len(exs):4d} examples | {has:4d} with corpus")


## Step 7 — Pre-Encode Passage Corpora

Building a FAISS index for every single question would be slow. Instead we **pre-encode all unique passages per dataset once** and cache embeddings. During evaluation, each question slices out its own passage vectors from this cache — making the main eval loop fast.


In [ ]:
from tqdm.auto import tqdm

PASSAGE_EMBEDDING_CACHE: Dict[str, Dict[str, np.ndarray]] = {}  # dataset → {passage_id: embedding}

for ds_name, examples in ALL_EXAMPLES.items():
    if not any(e.has_corpus for e in examples):
        print(f"[{ds_name}] No corpora — skipping pre-encode.")
        PASSAGE_EMBEDDING_CACHE[ds_name] = {}
        continue

    # Collect all unique passages across all examples
    unique_passages: Dict[str, str] = {}   # id → text
    for ex in examples:
        for p in ex.passages:
            if p["id"] not in unique_passages:
                unique_passages[p["id"]] = p["text"]

    print(f"[{ds_name}] Encoding {len(unique_passages):,} unique passages ...")

    ids   = list(unique_passages.keys())
    texts = [f"passage: {unique_passages[pid]}" for pid in ids]

    # Encode in batches of 512 to avoid OOM on Colab free tier
    all_embeds = []
    batch_size = 512
    for i in tqdm(range(0, len(texts), batch_size), desc=f"  {ds_name}"):
        batch = texts[i : i + batch_size]
        emb = ENCODER.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        all_embeds.append(emb)

    all_embeds = np.vstack(all_embeds).astype(np.float32)
    PASSAGE_EMBEDDING_CACHE[ds_name] = {pid: all_embeds[j] for j, pid in enumerate(ids)}
    print(f"  Done — embedding shape: {all_embeds.shape}")

print("\nPre-encoding complete.")


## Step 8 — Run Corpus-Conditioned Evaluation

For each `(dataset, mode)` pair the loop:
1. For each question → slice its passage embeddings from the cache → build tiny FAISS index
2. Run retrieval (`dense_only` / `bm25_only` / `hybrid` / `hybrid_rerank` with BGE cross-encoder)
3. Call the configured Llama model (via HuggingFace) with the top-K retrieved passages as context
4. Score EM, token F1, Recall@K, support rate, latency

`NQ` has no passage corpus — it is evaluated once with no-retrieval context, measuring generation quality only.


In [ ]:
    for i, q in enumerate(questions[:n_pilot]):
        print(f"  [{ds_name}] {i+1}/{n_pilot}: {q['question'][:70]}…")
        for mode in modes:
            try:
                rec = evaluate_question(
                    q["question"], q["passages"], q["gold_ids"], q["answer"],
                    mode, ds_name
                )
                records[mode].append(rec)
            except Exception as exc:
                print(f"    ERROR ({mode}): {exc}")
                rec = {"recall@5": 0, "supported": 0, "EM": 0, "F1": 0,
                       "pred": "", "gold": q["answer"]}
                records[mode].append(rec)

        # Print gold answer and one representative prediction (from hybrid_rerank or first mode)
        _sample = records.get("hybrid_rerank") or records.get("dense_only") or []
        _pred   = _sample[-1]["pred"] if _sample else ""
        print(f"    gold : {q['answer']}")
        print(f"    pred : {_pred}")
        time.sleep(0.05)


## Step 9 — Aggregate Results, Visualize, and Download


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ── Build flat summary DataFrame from ALL_RESULTS ─────────────────────────────
# ALL_RESULTS structure: {ds_name: {mode: {"recall@5", "supported", "EM", "F1"}}}

rows = []
for ds_name, mode_dict in ALL_RESULTS.items():
    for mode, metrics in mode_dict.items():
        rows.append({
            "dataset":    ds_name,
            "mode":       mode,
            "EM":         metrics.get("EM",        float("nan")),
            "F1":         metrics.get("F1",        float("nan")),
            "recall@5":   metrics.get("recall@5",  float("nan")),
            "supported":  metrics.get("supported", float("nan")),
        })

summary = pd.DataFrame(rows)
display(summary.sort_values(["dataset", "mode"]).to_string(index=False))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / "benchmark_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"\nSaved: {summary_path}")


# ── Bar charts ────────────────────────────────────────────────────────────────

METRICS = [
    ("EM",        "Exact Match (EM)"),
    ("F1",        "Token F1"),
    ("recall@5",  f"Recall@5"),
    ("supported", "Support Rate"),
]

MODE_ORDER = ["dense_only", "bm25_only", "hybrid", "hybrid_rerank", "no_retrieval"]
MODE_LABELS = {
    "dense_only":    "Dense\n(e5-base-v2)",
    "bm25_only":     "BM25\n(Okapi)",
    "hybrid":        "Hybrid\n(Fusion)",
    "hybrid_rerank": "Hybrid+\nRerank",
    "no_retrieval":  "No\nRetrieval",
}
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

# Only chart datasets that have retrieval baselines
retrieval_datasets = [d for d in summary["dataset"].unique() if d != "nq_open"]

if retrieval_datasets:
    n_rows = len(METRICS)
    n_cols = len(retrieval_datasets)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)

    for r, (metric_key, metric_label) in enumerate(METRICS):
        for c, ds in enumerate(retrieval_datasets):
            ax   = axes[r][c]
            sub  = summary[summary["dataset"] == ds]
            modes_present = [m for m in MODE_ORDER if m in sub["mode"].values]
            vals  = [float(sub.loc[sub["mode"] == m, metric_key].values[0])
                     if not pd.isna(sub.loc[sub["mode"] == m, metric_key].values[0]) else 0.0
                     for m in modes_present]
            labels = [MODE_LABELS.get(m, m) for m in modes_present]

            bars = ax.bar(labels, vals, color=COLORS[:len(modes_present)],
                          edgecolor="white", width=0.6)
            ax.set_title(f"{ds.upper()} — {metric_label}", fontsize=10, fontweight="bold")
            ax.set_ylabel(metric_label, fontsize=8)
            ax.set_ylim(0, max(vals) * 1.35 + 0.001)
            ax.tick_params(axis="x", labelsize=8)
            ax.tick_params(axis="y", labelsize=8)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.001,
                        f"{val:.4f}", ha="center", va="bottom", fontsize=7)

    plt.suptitle(
        f"PersonaRAG Corpus-Conditioned Baseline Comparison  (n={N_PILOT}/dataset)",
        fontsize=13, fontweight="bold", y=1.01,
    )
    plt.tight_layout()

    plot_dir = OUTPUT_DIR / "plots"
    plot_dir.mkdir(exist_ok=True)
    out_png = plot_dir / "baseline_comparison.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_png}")


# ── NQ generation-only results ────────────────────────────────────────────────

if "nq_open" in ALL_RESULTS:
    print("\nNQ Open — generation-only (no retrieval corpus available):")
    nq_df = pd.DataFrame([
        {"mode": mode, **metrics}
        for mode, metrics in ALL_RESULTS["nq_open"].items()
    ])
    display(nq_df.to_string(index=False))


# ── Download to local machine ─────────────────────────────────────────────────

try:
    from google.colab import files
    files.download(str(summary_path))
    if retrieval_datasets:
        files.download(str(out_png))
except Exception:
    print("(Not in Colab — files saved locally only.)")
